In [94]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspark-pilot')
    .master('spark://spark-master:7077')
    .config('spark.executor.instances', 1)  # Is ignored by spark standalone cluster manager
    .config('spark.cores.max', '4')         # Workaround to cap number of executor's instances
    .config('spark.executor.cores', 2)
    .config('spark.executor.memory', '1024M')
    .getOrCreate()
)

spark

In [95]:
schema = 'first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id string'
emp_df = spark.read.schema(schema).csv('/home/spark/data/employee/4', header=True)

In [96]:
emp_df.printSchema()

root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- department_id: string (nullable = true)



In [97]:
dept_names = {1 : 'Department 1', 
              2 : 'Department 2', 
              3 : 'Department 3', 
              4 : 'Department 4',
              5 : 'Department 5', 
              6 : 'Department 6', 
              7 : 'Department 7', 
              8 : 'Department 8', 
              9 : 'Department 9', 
              10 : 'Department 10'}

In [38]:
dept_schema = 'department_id integer, department string'
dept_df = spark.createDataFrame(data=list(dept_names.items()), schema=dept_schema)

In [39]:
emp_joined_df = emp_df.join(dept_df, on='department_id')

In [41]:
emp_joined_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [department_id#25, first_name#18, last_name#19, job_title#20, dob#21, email#22, phone#23, salary#24, department#27]
   +- SortMergeJoin [cast(department_id#25 as bigint)], [cast(department_id#26 as bigint)], Inner
      :- Sort [cast(department_id#25 as bigint) ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(cast(department_id#25 as bigint), 200), ENSURE_REQUIREMENTS, [plan_id=27]
      :     +- Filter isnotnull(department_id#25)
      :        +- FileScan csv [first_name#18,last_name#19,job_title#20,dob#21,email#22,phone#23,salary#24,department_id#25] Batched: false, DataFilters: [isnotnull(department_id#25)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/spark/data/employee/4], PartitionFilters: [], PushedFilters: [IsNotNull(department_id)], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
      +- Sort [cast(department_id#

In [43]:
emp_joined_df.write.mode('overwrite').format('noop').save()

In [44]:
broadcast_dept_names = spark.sparkContext.broadcast(dept_names)

In [45]:
broadcast_dept_names.value

{1: 'Department 1',
 2: 'Department 2',
 3: 'Department 3',
 4: 'Department 4',
 5: 'Department 5',
 6: 'Department 6',
 7: 'Department 7',
 8: 'Department 8',
 9: 'Department 9',
 10: 'Department 10'}

In [57]:
from pyspark.sql.functions import col, udf

@udf
def get_dept_names(id):
    return broadcast_dept_names.value.get(id)

In [58]:
broadcast_df = emp_df.withColumn('department_name', get_dept_names(col('department_id')))

In [59]:
broadcast_df.write.mode('overwrite').format('noop').save()

26/11/17 18:28:11 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 634844 ms exceeds timeout 120000 ms
26/11/17 18:28:11 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.5: Executor heartbeat timed out after 634844 ms


In [98]:
emp_df.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)900-5337|604738.0|            8|
|    Ashley|   Montoya|        C

In [114]:
number_of_richards = spark.sparkContext.accumulator(0)

def parse(row):
    if row['first_name'] == 'Felipe':
        number_of_richards.add(1)

In [115]:
emp_df.foreach(parse)

In [116]:
number_of_richards.value

0

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 50936)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/usr/local/lib/python3.10/dist-packages/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
  File "/usr/local/lib/python3.10/dist-packages/pyspark/accumulators.py", line 272, in poll
    if self.rfile in r and func():
  File "/usr/local/lib/python3.10/dist-packages/pyspark/accumulators.py", line 276, in accum_updates
    num_updates = read_int(